# 第7章 JSON 数据操作

## 📌 学习目标

- JSON 数据类型
- JSON 函数：提取、修改、查询
- JSON 索引
- JSON 聚合函数

---

In [ ]:
import os
import pymysql
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

DB_CONFIG = {
    "host": "localhost",
    "user": "root",
    "password": os.getenv("MYSQL_PASSWORD"),
    "database": "shop_db",
    "charset": "utf8mb4",
}

def run_query(sql, params=None):
    conn = pymysql.connect(**DB_CONFIG)
    try:
        return pd.read_sql(sql, conn, params=params)
    finally:
        conn.close()

## 1. JSON 数据类型

MySQL 5.7+ 支持原生 JSON 类型，可以存储和查询 JSON 文档。

```sql
CREATE TABLE t (
    id INT PRIMARY KEY,
    data JSON
);
```

In [ ]:
# 创建带 JSON 列的表
execute_sql("DROP TABLE IF EXISTS json_demo")
execute_sql("""
    CREATE TABLE json_demo (
        id INT PRIMARY KEY AUTO_INCREMENT,
        user_data JSON NOT NULL
    )
""")

# 插入 JSON 数据
execute_sql("""
    INSERT INTO json_demo (user_data) VALUES
    ('{"name": "张三", "age": 25, "hobbies": ["读书", "游泳"]}'),
    ('{"name": "李四", "age": 30, "hobbies": ["音乐", "旅行", "摄影"]}'),
    ('{"name": "王五", "age": 28, "hobbies": ["编程"]}')
""")

run_query("SELECT * FROM json_demo")

## 2. JSON 提取函数

| 函数 | 说明 | 示例 |
|------|------|------|
| `JSON_EXTRACT(json, path)` | 提取值 | `JSON_EXTRACT(data, '$.name')` |
| `->` | 提取（返回带引号） | `data->'$.name'` |
| `->>` | 提取（返回不带引号） | `data->>'$.name'` |
| `JSON_KEYS(json)` | 获取所有键 | `JSON_KEYS(data)` |
| `JSON_LENGTH(json)` | 获取长度 | `JSON_LENGTH(data)` |

In [ ]:
# JSON_EXTRACT 提取
run_query("""
    SELECT 
        JSON_EXTRACT(user_data, '$.name') AS name_with_quotes,
        user_data->>'$.name' AS name_clean,
        user_data->>'$.age' AS age,
        user_data->'$.hobbies' AS hobbies
    FROM json_demo
""")

In [ ]:
# 提取数组元素
run_query("""
    SELECT 
        user_data->>'$.name' AS name,
        user_data->'$.hobbies[0]' AS first_hobby,
        JSON_LENGTH(user_data->'$.hobbies') AS hobby_count
    FROM json_demo
""")

In [ ]:
# JSON_KEYS：获取所有键
run_query("SELECT JSON_KEYS(user_data) AS keys FROM json_demo LIMIT 1")

## 3. JSON 查询与过滤

In [ ]:
# WHERE 中使用 JSON 提取
run_query("""
    SELECT user_data->>'$.name' AS name, user_data->>'$.age' AS age
    FROM json_demo
    WHERE user_data->>'$.age' > '25'
""")

In [ ]:
# JSON_CONTAINS：检查数组是否包含某值
run_query("""
    SELECT user_data->>'$.name' AS name
    FROM json_demo
    WHERE JSON_CONTAINS(user_data->'$.hobbies', '"读书"')
""")

In [ ]:
# JSON_SEARCH：搜索值
run_query("""
    SELECT 
        user_data->>'$.name' AS name,
        JSON_SEARCH(user_data, 'one', '旅行') AS found_path
    FROM json_demo
    WHERE JSON_SEARCH(user_data, 'one', '旅行') IS NOT NULL
""")

## 4. JSON 修改函数

| 函数 | 说明 |
|------|------|
| `JSON_SET(json, path, val)` | 设置值（已存在则更新） |
| `JSON_INSERT(json, path, val)` | 插入值（已存在则忽略） |
| `JSON_REPLACE(json, path, val)` | 替换值（不存在则忽略） |
| `JSON_REMOVE(json, path)` | 删除值 |

In [ ]:
# JSON_SET：添加/更新字段
run_query("""
    SELECT 
        user_data->>'$.name' AS name,
        JSON_SET(user_data, '$.city', '北京') AS updated
    FROM json_demo
    WHERE id = 1
""")

In [ ]:
# JSON_ARRAY_APPEND：向数组追加元素
run_query("""
    SELECT 
        user_data->>'$.name' AS name,
        JSON_ARRAY_APPEND(user_data, '$.hobbies', '游泳') AS updated_hobbies
    FROM json_demo
    WHERE id = 3
""")

## 5. JSON 聚合函数

In [ ]:
# JSON_ARRAYAGG：将多行聚合为 JSON 数组
run_query("""
    SELECT 
        category_id,
        JSON_ARRAYAGG(product_name) AS products
    FROM products
    GROUP BY category_id
""")

In [ ]:
# JSON_OBJECTAGG：将多行聚合为 JSON 对象
run_query("""
    SELECT 
        category_id,
        JSON_OBJECTAGG(product_name, price) AS price_map
    FROM products
    GROUP BY category_id
""")

## 6. 清理

In [ ]:
execute_sql("DROP TABLE IF EXISTS json_demo")
print("清理完成")

## 📝 练习题

### 题目1（简单）
创建一个带 JSON 列的表，插入一条包含 name、email、address 的 JSON 数据，然后提取 name 和 email。

### 题目2（中等）
使用 JSON_ARRAYAGG 将每个分类的商品名称聚合成 JSON 数组。

### 题目3（中等）
使用 JSON_CONTAINS 查询 hobbies 中包含"读书"的用户。

---

## ✅ 参考答案

In [ ]:
# 题目1：JSON 基本操作
execute_sql("DROP TABLE IF EXISTS json_test")
execute_sql("CREATE TABLE json_test (id INT PRIMARY KEY AUTO_INCREMENT, info JSON)")
execute_sql("INSERT INTO json_test (info) VALUES ('{\"name\": \"测试用户\", \"email\": \"test@test.com\", \"address\": \"北京市\"}')")

run_query("""
    SELECT 
        info->>'$.name' AS name,
        info->>'$.email' AS email
    FROM json_test
""")
execute_sql("DROP TABLE IF EXISTS json_test")

In [ ]:
# 题目2：JSON_ARRAYAGG
run_query("""
    SELECT 
        c.category_name,
        JSON_ARRAYAGG(p.product_name) AS products
    FROM products p
    INNER JOIN categories c ON p.category_id = c.category_id
    GROUP BY c.category_name
""")

In [ ]:
# 题目3：JSON_CONTAINS
execute_sql("DROP TABLE IF EXISTS json_test")
execute_sql("CREATE TABLE json_test (id INT PRIMARY KEY AUTO_INCREMENT, data JSON)")
execute_sql("INSERT INTO json_test (data) VALUES ('{\"name\": \"张三\", \"hobbies\": [\"读书\", \"游泳\"]}')")
execute_sql("INSERT INTO json_test (data) VALUES ('{\"name\": \"李四\", \"hobbies\": [\"音乐\", \"旅行\"]}')")

run_query("""
    SELECT data->>'$.name' AS name
    FROM json_test
    WHERE JSON_CONTAINS(data->'$.hobbies', '"读书"')
""")
execute_sql("DROP TABLE IF EXISTS json_test")